<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Command-line interface

**[Command-line interface](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/)** · Python for Practical AI Engineering · Module 6, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** give ticket-cleaner a command-line interface with argparse (arguments, options with defaults, help text and clear exit codes), run the package with `python -m`, and test the whole command.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection: the setup installs httpx and pytest, and downloads the data files. No account. |
| **You should know** | `count_tickets.py` with logging and exit codes, from [Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/), and the tests from [Focused tests](https://learning.nextia-ai.com/courses/python/m05/focused-tests/). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here, with the tests. |
| **Tested** | 2026-10-07, Python 3.14.6 with httpx 0.28.1 and pytest 9.1.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M06-L02-command-line-interface/command-line-interface-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 and `pytest` 9.1.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`, `pytest 9.1.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1 pytest==9.1.1
import httpx; print("httpx", httpx.__version__)
import pytest; print("pytest", pytest.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, `tickets-excel.csv`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 19 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`), [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) (`remote.py` and `try_mock.py`), [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) (`config.py` and `.env.example`), [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/) (the two fixes in `parsing.py` and `files.py`), [Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/) (logging and exit codes), [Focused tests](https://learning.nextia-ai.com/courses/python/m05/focused-tests/) (the tests in `tests/`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
    "tickets-excel.csv": "cf0fbbce193bb1a595ce9207cc3f004aeedb118008e4ffa35585fda2f1167a95",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
import logging
from pathlib import Path

import httpx

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")

logger = logging.getLogger("ticket_cleaner")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> int:
    settings = load_settings()
    logging.basicConfig(
        level=settings.log_level, format="%(levelname)s %(name)s: %(message)s"
    )
    logger.info("reading tickets from %s", settings.source)

    try:
        rows = load_rows(settings)
    except (OSError, ValueError, httpx.HTTPError) as error:
        logger.error("cannot read the tickets from %s: %s", settings.source, error)
        return 1

    tickets, rejected = split_records(rows)
    for record in rejected:
        logger.warning("row %d rejected: %s", record.row, "; ".join(record.problems))

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    logger.info("wrote %s", OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
''',
    'requirements.txt': '''\
httpx==0.28.1
pytest==9.1.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    if not isinstance(raw, dict):
        return ["not a record"]
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8-sig", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
    'ticket_cleaner/remote.py': '''\
import httpx


def fetch_rows(
    url: str, token: str | None = None, client: httpx.Client | None = None
) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
''',
    'try_mock.py': '''\
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()
''',
    'ticket_cleaner/config.py': '''\
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)
    log_level: str


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
        log_level=os.environ.get("LOG_LEVEL", "WARNING").upper(),
    )
''',
    '.env.example': '''\
# Settings for ticket-cleaner. Set them as environment variables.
# TICKETS_SOURCE: a .csv or .json file, or "remote". Default: data/tickets.csv
TICKETS_SOURCE=data/tickets.csv
# TICKETS_URL: the web service for "remote". Default: the course's public file.
TICKETS_URL=
# TICKETS_API_TOKEN: secret. Never write a real token in this file.
TICKETS_API_TOKEN=
''',
    'data/broken.json': '''\
[
  {"id": "T-3001", "status": "open", "category": "billing", "priority": 2},
  null,
  {"id": "T-3003", "status": "closed", "category": "login", "priority": 3}
]
''',
    'tests/test_files.py': '''\
import json

from ticket_cleaner.files import read_rows, write_json


def test_csv_with_byte_order_mark(tmp_path):
    path = tmp_path / "tickets.csv"
    path.write_text("id,status\\nT-1001,open\\n", encoding="utf-8-sig")
    assert read_rows(path) == [{"id": "T-1001", "status": "open"}]


def test_write_json_makes_the_folder(tmp_path):
    path = tmp_path / "reports" / "summary.json"
    write_json({"selected": 0}, path)
    assert json.loads(path.read_text(encoding="utf-8")) == {"selected": 0}
    assert not (tmp_path / "reports" / "summary.json.tmp").exists()
''',
    'tests/test_parsing.py': '''\
import pytest

from ticket_cleaner.parsing import find_problems, split_records, to_ticket
from ticket_cleaner.records import Ticket


@pytest.fixture
def valid_raw():
    return {"id": "T-1001", "status": "open", "category": "billing", "priority": "2"}


def test_valid_record_has_no_problems(valid_raw):
    assert find_problems(valid_raw) == []


def test_text_is_cleaned(valid_raw):
    valid_raw["id"] = " t-1001 "
    valid_raw["status"] = "Open "
    expected = Ticket(id="T-1001", status="open", category="billing", priority=2)
    assert to_ticket(valid_raw) == expected


@pytest.mark.parametrize("priority", ["0", "4", "high", "2.5", "-1"])
def test_priority_outside_1_to_3_is_a_problem(valid_raw, priority):
    valid_raw["priority"] = priority
    assert find_problems(valid_raw) == [f"priority must be 1, 2 or 3, not {priority!r}"]


def test_missing_field_is_a_problem(valid_raw):
    del valid_raw["category"]
    assert find_problems(valid_raw) == ["missing category"]


def test_duplicate_id_is_rejected(valid_raw):
    tickets, rejected = split_records([valid_raw, dict(valid_raw)])
    assert len(tickets) == 1
    assert rejected[0].row == 2
    assert rejected[0].problems == ["duplicate id T-1001"]


def test_a_value_that_is_not_a_dictionary_is_rejected():
    tickets, rejected = split_records([None])
    assert tickets == []
    assert rejected[0].problems == ["not a record"]


def test_empty_input_gives_empty_results():
    assert split_records([]) == ([], [])
''',
    'tests/test_remote.py': '''\
import httpx
import pytest

from ticket_cleaner.remote import fetch_rows

URL = "https://tickets.example/api/tickets"


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def test_fetch_rows_returns_the_records():
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    assert fetch_rows(URL, client=client) == [{"id": "T-2001", "status": "open"}]


def test_fetch_rows_raises_on_server_error():
    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    with pytest.raises(httpx.HTTPStatusError):
        fetch_rows(URL, client=client)
''',
    'tests/test_report.py': '''\
import pytest

from ticket_cleaner.records import Ticket
from ticket_cleaner.report import average_priority, build_summary, count_by_category


@pytest.fixture
def tickets():
    return [
        Ticket(id="T-1", status="open", category="login", priority=1),
        Ticket(id="T-2", status="open", category="billing", priority=1),
        Ticket(id="T-3", status="closed", category="billing", priority=2),
    ]


def test_average_priority_of_no_tickets_is_none():
    assert average_priority([]) is None


def test_average_priority_is_rounded_to_one_decimal(tickets):
    assert average_priority(tickets) == 1.3


def test_categories_are_in_alphabetical_order(tickets):
    assert list(count_by_category(tickets)) == ["billing", "login"]


def test_summary_for_a_status_with_no_tickets(tickets):
    summary = build_summary(tickets, [], status="pending")
    assert summary["selected"] == 0
    assert summary["by_category"] == {}
    assert summary["average_priority"] is None
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- `tree()` shows the files and folders of the project, like the folder trees on the lesson page.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

def tree():
    """Show the files and folders of the project, one level of indent per folder."""
    print("ticket-cleaner/")
    for path in sorted(PROJECT.rglob("*")):
        parts = path.relative_to(PROJECT).parts
        if any(part in ("__pycache__", ".pytest_cache") for part in parts):
            continue
        print("    " * len(parts) + path.name + ("/" if path.is_dir() else ""))

import tempfile
from pathlib import Path

def check_strict():
    """Run the command with and without --strict, then the tests."""
    with tempfile.TemporaryDirectory() as folder:
        folder = Path(folder)
        clean = folder / "clean.csv"
        clean.write_text("id,status,category,priority\nT-1,open,login,1\n", encoding="utf-8")
        cases = [
            (["data/tickets.csv", "--output", str(folder / "a.json")], 0, "without --strict, the exit code must not change"),
            (["data/tickets.csv", "--strict", "--output", str(folder / "b.json")], 3, "with --strict and rejected records"),
            ([str(clean), "--strict", "--output", str(folder / "c.json")], 0, "with --strict and clean data"),
        ]
        for arguments, expected, situation in cases:
            result = run_quiet("-m", "ticket_cleaner", *arguments)
            if result.returncode != expected:
                last = (result.stderr.strip().splitlines() or [""])[-1]
                print(f"Not yet: {situation}, the exit code is {result.returncode}. {last}")
                return
        if not (folder / "b.json").exists():
            print("Not yet: with --strict, the program must still write the report.")
            return
    if "--strict" not in run_quiet("-m", "ticket_cleaner", "--help").stdout:
        print("Not yet: --help does not show --strict. Give the option a help text.")
        return
    tests = run_quiet("-m", "pytest", "-q", "-k", "strict")
    if tests.returncode != 0:
        print("Not yet: add a test for each case, with strict in its name, and make the tests pass.")
        return
    print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Arguments: what changes on each run

Your own program can take **arguments** and **options**, as `cd projects` and `python3 --version` do. Python puts the words of the command into a list, `sys.argv`. Run the two cells: the first saves a small script, the second runs it as `python show_args.py data/tickets.csv --status pending`.

In [ ]:
%%writefile show_args.py
import sys

print(sys.argv)

In [ ]:
run("show_args.py", "data/tickets.csv", "--status", "pending")

> **Check.** You should see `['show_args.py', 'data/tickets.csv', '--status', 'pending']`. The first item is the script, and the rest are the words that you typed, as strings.

The `argparse` module reads this list for you, and also handles a missing value, a typing mistake in an option, and the help text. Which settings become arguments? Ask: **does it change on each run, or does it belong to the place where the program runs?** The input file and the status change on each run: an argument and an option. The service URL, the token and the log level stay environment variables.

## 2. Worked example: argparse and python -m

A package can run as a program if it has a file `__main__.py`: `python -m ticket_cleaner` runs that file. The program's code goes in a new module, `cli.py`. `__main__.py` only starts it. Run the two cells.

In [ ]:
%%writefile ticket_cleaner/cli.py
import argparse
import logging
from pathlib import Path

import httpx

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.records import STATUSES
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

logger = logging.getLogger("ticket_cleaner")


def make_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="ticket_cleaner",
        description="Check support-ticket records and write a summary report.",
    )
    parser.add_argument(
        "input", help="a .csv or .json file of tickets, or 'remote' to download them"
    )
    parser.add_argument(
        "--status",
        default="open",
        choices=sorted(STATUSES),
        help="which tickets to count (default: open)",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("reports/summary.json"),
        help="where to save the report (default: reports/summary.json)",
    )
    return parser


def load_rows(source: str, settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(source))


def main(argv: list[str] | None = None) -> int:
    args = make_parser().parse_args(argv)
    settings = load_settings()
    logging.basicConfig(
        level=settings.log_level, format="%(levelname)s %(name)s: %(message)s"
    )
    logger.info("reading tickets from %s", args.input)

    try:
        rows = load_rows(args.input, settings)
    except (OSError, ValueError, httpx.HTTPError) as error:
        logger.error("cannot read the tickets from %s: %s", args.input, error)
        return 1

    tickets, rejected = split_records(rows)
    for record in rejected:
        logger.warning("row %d rejected: %s", record.row, "; ".join(record.problems))

    summary = build_summary(tickets, rejected, args.status)
    write_json(summary, args.output)
    logger.info("wrote %s", args.output)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {args.output}")
    return 0

In [ ]:
%%writefile ticket_cleaner/__main__.py
from ticket_cleaner.cli import main

raise SystemExit(main())

In `make_parser`, `"input"` is a **positional argument**: required, and identified by its place. `"--status"` is an **option** with a default and `choices`. `type=Path` converts the text to a `Path`. `main(argv=None)` reads `sys.argv`, and a test can pass its own list instead.

The input is now an argument, so `config.py` loses its `source` field. And `count_tickets.py` is no longer needed: its job is now in `cli.py`. Run the two cells.

In [ ]:
%%writefile ticket_cleaner/config.py
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    tickets_url: str
    api_token: str | None = field(repr=False)
    log_level: str


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
        log_level=os.environ.get("LOG_LEVEL", "WARNING").upper(),
    )

In [ ]:
from pathlib import Path

Path("count_tickets.py").unlink(missing_ok=True)
tree()

> **Check.** The tree shows no `count_tickets.py`, and `ticket_cleaner/` has `__main__.py` and `cli.py`.

## 3. Guided practice: from script to command

Show the help that argparse makes from the parser. Run the cell.

In [ ]:
run("-m", "ticket_cleaner", "--help")

> **Check.** You should see `usage: ticket_cleaner [-h] [--status {closed,open,pending}] [--output OUTPUT]`, the description, the positional argument `input` and the three options, each with its help text.

Run it with the defaults, then for the pending tickets from the JSON file, into a different report.

In [ ]:
run("-m", "ticket_cleaner", "data/tickets.csv")
run("-m", "ticket_cleaner", "data/tickets.json", "--status", "pending", "--output", "reports/pending.json")

> **Check.** Each run ends with `15 rows: 9 valid, 6 rejected.`, the `Report:` line and `(exit code 0)`. `reports/pending.json` has `"selected": 2`.

The program now has three exit codes: `0` success (rejected records are a normal result), `1` the input could not be read, and `2` the command is wrong. `argparse` prints the usage and the error, and exits with `2` for you.

> **Predict.** Which exit code does each command give? Then run the cell.

In [ ]:
run("-m", "ticket_cleaner", "data/nothing.csv")
run("-m", "ticket_cleaner", "data/tickets.csv", "--status", "Pending")

> **Check.** The first shows `ERROR ticket_cleaner: cannot read the tickets from data/nothing.csv: …` and `(exit code 1)`. The second shows the usage, then `ticket_cleaner: error: argument --status: invalid choice: 'Pending' (choose from 'closed', 'open', 'pending')`, and `(exit code 2)`.

## 4. Test the command

Because `main` accepts a list, a test can run the whole program, and send the report to `tmp_path`. Run the cell, then all the tests.

In [ ]:
%%writefile tests/test_cli.py
import json

from ticket_cleaner.cli import main


def test_report_for_pending_tickets(tmp_path):
    output = tmp_path / "pending.json"
    exit_code = main(
        ["data/tickets.csv", "--status", "pending", "--output", str(output)]
    )
    assert exit_code == 0
    summary = json.loads(output.read_text(encoding="utf-8"))
    assert summary["selected"] == 2
    assert len(summary["rejected"]) == 6


def test_missing_file_gives_exit_code_1(tmp_path):
    assert main([str(tmp_path / "missing.csv")]) == 1

In [ ]:
run("-m", "pytest", "-q")

> **Check.** You should see `21 passed`: the 19 tests of Module 5 and the 2 new ones.

## 5. Your turn: a --strict option

The help desk lead wants to use ticket-cleaner in an automatic job, which must stop when the data has problems.

> **Your turn.** Add an option `--strict` to `cli.py` in the next cell. With it, the program still writes the report, but ends with exit code `3` if any record was rejected. Without it, the behaviour does not change. Use `action="store_true"`: the option takes no value, and `args.strict` is `True` when the user gives it. Give it a help text.
>
> Then add a test for each case to `tests/test_cli.py` in the cell after it, with `strict` in each test name. Run both cells, the `run` cell, and the check cell.

In [ ]:
%%writefile ticket_cleaner/cli.py
import argparse
import logging
from pathlib import Path

import httpx

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.records import STATUSES
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

logger = logging.getLogger("ticket_cleaner")


def make_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog="ticket_cleaner",
        description="Check support-ticket records and write a summary report.",
    )
    parser.add_argument(
        "input", help="a .csv or .json file of tickets, or 'remote' to download them"
    )
    parser.add_argument(
        "--status",
        default="open",
        choices=sorted(STATUSES),
        help="which tickets to count (default: open)",
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("reports/summary.json"),
        help="where to save the report (default: reports/summary.json)",
    )
    parser.add_argument(
        "--strict",
        action="store_true",
        help="end with exit code 3 if any record is rejected",
    )
    return parser


def load_rows(source: str, settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(source))


def main(argv: list[str] | None = None) -> int:
    args = make_parser().parse_args(argv)
    settings = load_settings()
    logging.basicConfig(
        level=settings.log_level, format="%(levelname)s %(name)s: %(message)s"
    )
    logger.info("reading tickets from %s", args.input)

    try:
        rows = load_rows(args.input, settings)
    except (OSError, ValueError, httpx.HTTPError) as error:
        logger.error("cannot read the tickets from %s: %s", args.input, error)
        return 1

    tickets, rejected = split_records(rows)
    for record in rejected:
        logger.warning("row %d rejected: %s", record.row, "; ".join(record.problems))

    summary = build_summary(tickets, rejected, args.status)
    write_json(summary, args.output)
    logger.info("wrote %s", args.output)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {args.output}")
    if args.strict and rejected:
        return 3
    return 0

In [ ]:
%%writefile tests/test_cli.py
import json

from ticket_cleaner.cli import main


def test_report_for_pending_tickets(tmp_path):
    output = tmp_path / "pending.json"
    exit_code = main(
        ["data/tickets.csv", "--status", "pending", "--output", str(output)]
    )
    assert exit_code == 0
    summary = json.loads(output.read_text(encoding="utf-8"))
    assert summary["selected"] == 2
    assert len(summary["rejected"]) == 6


def test_missing_file_gives_exit_code_1(tmp_path):
    assert main([str(tmp_path / "missing.csv")]) == 1


def test_strict_with_rejected_records_gives_exit_code_3(tmp_path):
    output = tmp_path / "summary.json"
    assert main(["data/tickets.csv", "--strict", "--output", str(output)]) == 3
    assert output.exists()


def test_strict_with_clean_data_gives_exit_code_0(tmp_path):
    clean = tmp_path / "clean.csv"
    clean.write_text("id,status,category,priority\nT-1,open,login,1\n", encoding="utf-8")
    assert main([str(clean), "--strict", "--output", str(tmp_path / "s.json")]) == 0

In [ ]:
run("-m", "ticket_cleaner", "data/tickets.csv", "--strict")
run("-m", "pytest", "-q")

In [ ]:
check_strict()

When it is correct, the run with `--strict` ends with `(exit code 3)`, and the tests show `23 passed`. Why `3` and not `1`? Exit code `1` already means "could not read the input". A different code lets the automatic job tell the two cases apart.

## 6. Failure case: No module named ticket_cleaner.__main__

Tomás names the file `main.py` instead of `__main__.py`.

> **Predict.** What does `python -m ticket_cleaner` say? Run the cell. It renames the file, runs the command, and renames the file back.

In [ ]:
Path("ticket_cleaner/__main__.py").rename("ticket_cleaner/main.py")
run("-m", "ticket_cleaner", "data/tickets.csv")
Path("ticket_cleaner/main.py").rename("ticket_cleaner/__main__.py")
print("__main__.py is back.")

> **Check.** You should see `No module named ticket_cleaner.__main__; 'ticket_cleaner' is a package and cannot be directly executed`, and `(exit code 1)`.

**Fix:** list the package folder. It must contain `__main__.py`, with two `_` before and after `main`. If the message is `No module named ticket_cleaner`, the terminal is not in the project folder, or the environment is not active.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/). They count toward your certificate when you are signed in and enrolled. On your computer, make the same changes in your project, and check the three exit codes with `echo $?` (or `$LASTEXITCODE`), as in the guided practice of the lesson.

In this lesson you learned that `argparse` describes a command with arguments, options, defaults, choices and help, that values which change on each run are arguments while secrets stay in environment variables, that exit codes say what happened, and that `__main__.py` makes a package run with `python -m`.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Quality and handoff](https://learning.nextia-ai.com/courses/python/m06/quality-and-handoff/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m06/command-line-interface/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.